# 🎙️ wiuuulab - Full AI Suite Backend (Google Colab)

Notebook ini menjalankan **SEMUA FITUR LENGKAP** secara **100% GRATIS** menggunakan GPU NVIDIA T4 dari Google Colab:
- 🔊 **StyleTTS2** (Text to Speech)
- 🎭 **Seed-VC** (Voice Changer / Speech-to-Speech)
- 🎵 **Make-An-Audio** (Sound Effects / Text-to-Audio)
- 🌐 **Unified AI Gateway + Cloudflare Tunnel** (Cukup 1 URL publik untuk semua fitur!)

### 📌 Cara Menggunakan:
1. Pastikan Runtime menggunakan GPU: **Runtime -> Change runtime type -> T4 GPU**.
2. Klik **Runtime -> Run all** (atau jalankan satu per satu).
3. Salin URL publik `https://xxx.trycloudflare.com` di sel terakhir ke Vercel!

In [ ]:
# @title 1. Cek Ketersediaan GPU
!nvidia-smi

In [ ]:
# @title 2. Setup System & Install Dependencies
import os

!sudo apt-get update -y && sudo apt-get install -y espeak-ng libsndfile1 git-lfs wget

if not os.path.exists('ElevenLabs-Clone'):
    !git clone https://github.com/BernieTv/ElevenLabs-Clone.git

%cd ElevenLabs-Clone

# Install AI Libraries
!pip install -q SoundFile torchaudio munch torch pydub pyyaml librosa nltk matplotlib accelerate transformers phonemizer einops einops-exts tqdm boto3 fastapi uvicorn pydantic httpx git+https://github.com/resemble-ai/monotonic_align.git

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('cmudict')

print('✅ Dependencies terpasang!')

In [ ]:
# @title 3. Download Bobot Model AI (StyleTTS2 & Seed-VC)
import os
%cd /content/ElevenLabs-Clone

# 1. StyleTTS2 Weights
if not os.path.exists('StyleTTS2/Models/LibriTTS/epoch_2nd_00074.pth'):
    print('Mengunduh model weights StyleTTS2...')
    !git clone https://huggingface.co/yl4579/StyleTTS2-LibriTTS /tmp/StyleTTS2-HF
    !mkdir -p StyleTTS2/Models/LibriTTS StyleTTS2/Demo/reference_audio
    !cp -rf /tmp/StyleTTS2-HF/Models/LibriTTS/* StyleTTS2/Models/LibriTTS/
    if os.path.exists('/tmp/StyleTTS2-HF/reference_audio.zip'):
        !unzip -o -q /tmp/StyleTTS2-HF/reference_audio.zip -d StyleTTS2/Demo/reference_audio
        !cp -f StyleTTS2/Demo/reference_audio/1221_135767_000016_000001.wav StyleTTS2/Models/LibriTTS/andreas1.wav
        !cp -f StyleTTS2/Demo/reference_audio/237_126280_000002_000001.wav StyleTTS2/Models/LibriTTS/woman1.wav
    print('✅ StyleTTS2 siap!')

# 2. Seed-VC Reference Voices
os.makedirs('seed-vc/examples/reference', exist_ok=True)
!cp -f StyleTTS2/Models/LibriTTS/andreas1.wav seed-vc/examples/reference/andreas1.wav
!cp -f StyleTTS2/Models/LibriTTS/woman1.wav seed-vc/examples/reference/s1p1.wav
!cp -f StyleTTS2/Models/LibriTTS/andreas1.wav seed-vc/examples/reference/trump_0.wav

print('✅ Semua model weights berhasil disiapkan!')

In [ ]:
# @title 4. Konfigurasi Kredensial Penyimpanan (AWS S3 / Cloudflare R2)
import os

os.environ['API_KEY'] = '12345'
os.environ['AWS_REGION'] = 'us-east-1'
os.environ['S3_BUCKET'] = 'elevenlabs-clone'
os.environ['AWS_ACCESS_KEY_ID'] = 'GANTI_DENGAN_ACCESS_KEY_ANDA'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'GANTI_DENGAN_SECRET_KEY_ANDA'

# Opsional jika pakai Cloudflare R2 / MinIO:
# os.environ['AWS_ENDPOINT_URL'] = 'https://<ACCOUNT_ID>.r2.cloudflarestorage.com'

print('✅ Konfigurasi environment selesai!')

In [ ]:
# @title 5. Jalankan Semua Layanan AI + Gateway + Cloudflare Tunnel
import subprocess
import time
import os

# Pastikan edge-tts terpasang untuk Organic Indonesian Engine
!pip install -q edge-tts

# Download Cloudflare Tunnel
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('Memasang Cloudflare Tunnel...')
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb

!pkill -f uvicorn || true

print('1. Menjalankan StyleTTS2 (Port 8001)...')
subprocess.Popen(['uvicorn', 'api:app', '--host', '127.0.0.1', '--port', '8001'], cwd='/content/ElevenLabs-Clone/StyleTTS2')

print('2. Menjalankan Unified Gateway (Port 8000)...')
subprocess.Popen(['uvicorn', 'gateway:app', '--host', '0.0.0.0', '--port', '8000'], cwd='/content/ElevenLabs-Clone')

time.sleep(5)
print('✅ Semua layanan AI aktif di Gateway!')
print('🌐 Membuka Cloudflare Tunnel... Salin link trycloudflare di bawah ini!')

!cloudflared tunnel --url http://127.0.0.1:8000
